# Project Cortex Model - Neo4j Graph Creation

This notebook reads Project objects from BAML's `ExtractProject` function and creates corresponding Neo4j nodes and relationships.

In [1]:
import os
import json
import getpass
from dotenv import load_dotenv
from pathlib import Path
from typing import Optional

# Load environment variables
load_dotenv('.env', override=True)

if not os.environ.get('NEO4J_URI'):
    os.environ['NEO4J_URI'] = getpass.getpass('NEO4J_URI:\n')
if not os.environ.get('NEO4J_USERNAME'):
    os.environ['NEO4J_USERNAME'] = getpass.getpass('NEO4J_USERNAME:\n')
if not os.environ.get('NEO4J_PASSWORD'):
    os.environ['NEO4J_PASSWORD'] = getpass.getpass('NEO4J_PASSWORD:\n')

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

# Import BAML client and types
from baml_client import b
from baml_client.types import Project

# Import Neo4j driver
from neo4j import GraphDatabase, RoutingControl

In [2]:
# Initialize Neo4j driver
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

# Test connection
result = driver.execute_query("MATCH(n) RETURN count(n) as node_count")
print(f"Connected to Neo4j. Total nodes: {result.records[0]['node_count']}")

Connected to Neo4j. Total nodes: 625


## Function to Create Project Graph in Neo4j

This function takes a Project object and creates all nodes and relationships according to the schema.

In [3]:
def create_project_graph(project: Project, driver: GraphDatabase.driver):
    """
    Create Neo4j nodes and relationships from a Project object.
    
    Args:
        project: Project object from BAML ExtractProject function
        driver: Neo4j GraphDatabase driver instance
    """
    
    with driver.session() as session:
        # Create Project node
        session.execute_write(_create_project_node, project)
        
        # Create Purpose and Objectives
        session.execute_write(_create_purpose_and_objectives, project)
        
        # Create Value, Benefits, and Metrics
        session.execute_write(_create_value_and_benefits, project)
        
        # Create Outcomes, SuccessCriteria, and MeasurableResults
        session.execute_write(_create_outcomes, project)
        
        # Create Technologies
        session.execute_write(_create_technologies, project)
        
        # Create Approach, Plan, Methods, Tools, Timeline, and Milestones
        session.execute_write(_create_approach_and_plan, project)
        
        # Create Team, Roles, Responsibilities, and Members
        session.execute_write(_create_team_and_roles, project)
        
    print(f"✓ Successfully created graph for project: {project.name}")

In [4]:
def _create_project_node(tx, project: Project):
    """Create the main Project node."""
    tx.run("""
        MERGE (p:Project {name: $name})
        SET p.description = $description,
            p.employer = $employer,
            p.location = $location,
            p.startDate = $startDate,
            p.endDate = $endDate,
            p.duration = $duration
        RETURN p
    """, name=project.name, 
           description=project.description,
           employer=project.employer,
           location=project.location,
           startDate=project.startDate,
           endDate=project.endDate,
           duration=project.duration)


def _create_purpose_and_objectives(tx, project: Project):
    """Create Purpose node and Objectives, link to Project."""
    # Create Purpose node
    tx.run("""
        MATCH (p:Project {name: $project_name})
        MERGE (purpose:Purpose {description: $description})
        MERGE (p)-[:HAS_PURPOSE]->(purpose)
    """, project_name=project.name, description=project.purpose.description)
    
    # Create Objectives nodes
    for objective in project.purpose.objectives:
        tx.run("""
            MATCH (purpose:Purpose {description: $description})
            MERGE (obj:Objective {text: $objective})
            MERGE (purpose)-[:HAS_OBJECTIVE]->(obj)
        """, description=project.purpose.description, objective=objective)


def _create_value_and_benefits(tx, project: Project):
    """Create Value node, Benefits, and Metrics, link to Project."""
    # Create Value node
    tx.run("""
        MATCH (p:Project {name: $project_name})
        MERGE (v:Value {description: $description})
        MERGE (p)-[:DELIVERS]->(v)
    """, project_name=project.name, description=project.value.description)
    
    # Create Benefits nodes
    for benefit in project.value.benefits:
        tx.run("""
            MATCH (v:Value {description: $description})
            MERGE (b:Benefit {text: $benefit})
            MERGE (v)-[:HAS_BENEFIT]->(b)
        """, description=project.value.description, benefit=benefit)
    
    # Create Metrics nodes (if they exist)
    if project.value.metrics:
        for metric in project.value.metrics:
            tx.run("""
                MATCH (v:Value {description: $description})
                MERGE (m:Metric {text: $metric})
                MERGE (v)-[:MEASURED_BY]->(m)
            """, description=project.value.description, metric=metric)


def _create_outcomes(tx, project: Project):
    """Create Outcomes, SuccessCriteria, and MeasurableResults nodes."""
    # Link Purpose to Outcomes
    tx.run("""
        MATCH (purpose:Purpose {description: $description})
        WITH purpose
        UNWIND $outcomes AS outcome_desc
        MERGE (outcome:Outcome {description: outcome_desc})
        MERGE (purpose)-[:DEFINES_OUTCOMES]->(outcome)
    """, description=project.purpose.description, 
           outcomes=[outcome.description for outcome in project.outcomes])
    
    # Create SuccessCriteria and MeasurableResults for each Outcome
    for outcome in project.outcomes:
        # Create SuccessCriteria
        for criterion in outcome.successCriteria:
            tx.run("""
                MATCH (outcome:Outcome {description: $outcome_desc})
                MERGE (crit:SuccessCriteria {text: $criterion})
                MERGE (outcome)-[:HAS_CRITERIA]->(crit)
            """, outcome_desc=outcome.description, criterion=criterion)
        
        # Create MeasurableResults (if they exist)
        if outcome.measurableResults:
            for result in outcome.measurableResults:
                tx.run("""
                    MATCH (outcome:Outcome {description: $outcome_desc})
                    MERGE (mr:MeasurableResult {text: $result})
                    MERGE (outcome)-[:HAS_RESULTS]->(mr)
                """, outcome_desc=outcome.description, result=result)


def _create_technologies(tx, project: Project):
    """Create Technology nodes and link to Project."""
    for tech in project.technologies:
        tx.run("""
            MATCH (p:Project {name: $project_name})
            MERGE (t:Technology {name: $name})
            SET t.description = $description
            MERGE (p)-[:USES_TECH]->(t)
        """, project_name=project.name, name=tech.name, description=tech.description)


def _create_approach_and_plan(tx, project: Project):
    """Create Approach, Plan, Methods, Tools, Timeline, and Milestones."""
    approach = project.approach
    plan = approach.plan
    
    # Create Approach node and link to Project
    tx.run("""
        MATCH (p:Project {name: $project_name})
        MERGE (a:Approach {description: $description})
        SET a.methodology = $methodology
        MERGE (p)-[:USES_APPROACH]->(a)
    """, project_name=project.name, 
           description=approach.description, 
           methodology=approach.methodology)
    
    # Create Plan node and link to Approach
    tx.run("""
        MATCH (a:Approach {description: $approach_desc})
        MERGE (plan:Plan {name: $plan_name})
        SET plan.description = $plan_description
        MERGE (a)-[:HAS_PLAN]->(plan)
    """, approach_desc=approach.description,
           plan_name=plan.name,
           plan_description=plan.description)
    
    # Create Methods and MethodSteps
    for method in plan.methods:
        tx.run("""
            MATCH (plan:Plan {name: $plan_name})
            MERGE (m:Method {name: $method_name})
            SET m.description = $method_description
            MERGE (plan)-[:USES_METHOD]->(m)
        """, plan_name=plan.name,
               method_name=method.name,
               method_description=method.description)
        
        # Create MethodSteps (if they exist)
        if method.steps:
            for step in method.steps:
                tx.run("""
                    MATCH (m:Method {name: $method_name})
                    MERGE (step:MethodStep {text: $step})
                    MERGE (m)-[:HAS_STEP]->(step)
                """, method_name=method.name, step=step)
    
    # Create Tools
    for tool in plan.tools:
        tx.run("""
            MATCH (plan:Plan {name: $plan_name})
            MERGE (tool:Tool {name: $tool_name})
            SET tool.description = $tool_description,
                tool.category = $category
            MERGE (plan)-[:USES_TOOL]->(tool)
        """, plan_name=plan.name,
               tool_name=tool.name,
               tool_description=tool.description,
               category=tool.category)
    
    # Create Timeline (if it exists)
    if plan.timeline:
        tx.run("""
            MATCH (plan:Plan {name: $plan_name})
            MERGE (timeline:Timeline {text: $timeline})
            MERGE (plan)-[:HAS_TIMELINE]->(timeline)
        """, plan_name=plan.name, timeline=plan.timeline)
    
    # Create Milestones (if they exist)
    if plan.milestones:
        for milestone in plan.milestones:
            tx.run("""
                MATCH (plan:Plan {name: $plan_name})
                MERGE (milestone:Milestone {text: $milestone})
                MERGE (plan)-[:HAS_MILESTONES]->(milestone)
            """, plan_name=plan.name, milestone=milestone)


def _create_team_and_roles(tx, project: Project):
    """Create Team, Roles, Responsibilities, and TeamMembers."""
    team = project.team
    
    # Create Team node and link to Project
    tx.run("""
        MATCH (p:Project {name: $project_name})
        MERGE (t:Team {name: $team_name})
        SET t.description = $team_description
        MERGE (p)-[:INVOLVES]->(t)
    """, project_name=project.name,
           team_name=team.name,
           team_description=team.description)
    
    # Create Roles and Responsibilities
    for role in team.roles:
        tx.run("""
            MATCH (t:Team {name: $team_name})
            MERGE (r:Role {name: $role_name})
            SET r.description = $role_description
            MERGE (t)-[:HAS_ROLE]->(r)
        """, team_name=team.name,
               role_name=role.name,
               role_description=role.description)
        
        # Create Responsibilities for each Role
        for resp in role.responsibilities:
            tx.run("""
                MATCH (r:Role {name: $role_name})
                MERGE (resp:Responsibility {description: $resp_desc})
                MERGE (r)-[:RESPONSIBLE_FOR]->(resp)
            """, role_name=role.name, resp_desc=resp.description)
            
            # Create Tasks (if they exist)
            if resp.tasks:
                for task in resp.tasks:
                    tx.run("""
                        MATCH (resp:Responsibility {description: $resp_desc})
                        MERGE (task:Task {text: $task})
                        MERGE (resp)-[:HAS_TASK]->(task)
                    """, resp_desc=resp.description, task=task)
            
            # Create Deliverables (if they exist)
            if resp.deliverables:
                for deliverable in resp.deliverables:
                    tx.run("""
                        MATCH (resp:Responsibility {description: $resp_desc})
                        MERGE (del:Deliverable {text: $deliverable})
                        MERGE (resp)-[:HAS_DELIVERABLE]->(del)
                    """, resp_desc=resp.description, deliverable=deliverable)
    
    # Create TeamMembers (if they exist)
    if team.members:
        for member in team.members:
            tx.run("""
                MATCH (t:Team {name: $team_name})
                MERGE (member:TeamMember {name: $member})
                MERGE (t)-[:INCLUDES]->(member)
            """, team_name=team.name, member=member)

## Usage Examples

### Example 1: Load Project from JSON file

In [ ]:
# Load Project from JSON file created by project_extractor.py
from pathlib import Path


project_data_dir = Path('project_data/input')
#son_files = list[Path](project_data_dir.glob('*.json'))
filename = 'project_data/input/InstrumentOnDataMesh.txt'
json_files = [filename]

if json_files:
    # Load the first JSON file as an example
    with open(json_files[0], 'r', encoding='utf-8') as f:
        project_dict = json.load(f)
    
    # Convert dict to Project object
    project = Project(**project_dict)
    
    # Create graph in Neo4j
    create_project_graph(project, driver)
else:
    print("No JSON files found in project_data directory")

### Example 2: Extract Project from raw content and create graph

In [ ]:
# Example: Extract project from raw content
project_content = """
Project: Metadata Agent
Description: A system to enable intuitive access to data catalogs through natural language queries.
"""

# Extract project using BAML
project = b.ExtractProject(project_content)

# Create graph in Neo4j
create_project_graph(project, driver)

### Example 3: Extract Project from a file with raw project content and create graph

In [5]:
from project_extractor import get_project_content, extract_project, write_project_to_file
# Create graph in Neo4j
from fileinput import filename
import sys

filename = 'project_data/input/InstrumentOnDataMesh.txt'
project_content = get_project_content(filename)
        
if not project_content.strip():
    print("Error: File is empty", file=sys.stderr)
    sys.exit(1)
        
 # Extract project information
project = extract_project(project_content)
print(f"\n✓ Successfully extracted project: {project.name}")

create_project_graph(project, driver)
print(f"\n✓ Successfully created graph for project: {project.name}")
 
 # Write to fileclear
output_path = write_project_to_file(project)
        
print(f"\n✓ Successfully wrote serializable project: {project.name} to Output file: {output_path}")

Extracting project information...
2025-12-15T17:10:00.225 [BAML INFO] Function ExtractProject:
    Client: CustomGPT4oMini (gpt-4o-mini-2024-07-18) - 19153ms. StopReason: stop. Tokens(in/out): 933/948
    ---PROMPT---
    system: Extract project information from this content, organizing it into the Why (Purpose & Value), How (Approach & Plan), and Who (Team & Responsibilities) structure:
    Project:
    - Name: Instrument On DataNMesh
    - Lengthy name: Instrument Data on DataMesh - Publishing Pipeline
    - Team, Employer, Location: Asset & Wealth Management, JPMorgan Chase, Jersey City, NJ
    - Role held: Lead Software Engineer, VP
    - Description:
    1) Led the instrument reference data to Data Mesh initiative as both technical lead and hands-on engineer
    2)
    Situation: Prior to DataMesh implementation, each AWM (Asset and Wealth Management) team and organization-wide consumer had bespoke integrations with Instrument data sources for their individual w✓ Project extractio

## Query Examples

### Find all projects and their purposes

In [ ]:
result = driver.execute_query("""
    MATCH (p:Project)-[:HAS_PURPOSE]->(purpose:Purpose)
    RETURN p.name as project_name, purpose.description as purpose
""")

for record in result.records:
    print(f"Project: {record['project_name']}")
    print(f"Purpose: {record['purpose']}")
    print("---")

In [ ]:
# Find teams and their roles for a specific project
result = driver.execute_query("""
    MATCH (p:Project {name: $project_name})-[:INVOLVES]->(t:Team)-[:HAS_ROLE]->(r:Role)
    RETURN p.name as project, t.name as team, r.name as role, r.description as role_description
    ORDER BY team, role
""", project_name="Metadata Agent")

for record in result.records:
    print(f"Project: {record['project']}")
    print(f"Team: {record['team']}")
    print(f"Role: {record['role']}")
    print(f"Description: {record['role_description']}")
    print("---")

In [ ]:
# Find all technologies used by projects
result = driver.execute_query("""
    MATCH (p:Project)-[:USES_TECH]->(t:Technology)
    RETURN p.name as project, t.name as technology, t.description as tech_description
    ORDER BY project, technology
""")

for record in result.records:
    print(f"{record['project']} uses {record['technology']}: {record['tech_description']}")

In [ ]:
# Close the driver connection
driver.close()